# Interactive CQI Visualizations

Plotly-based exploration of coffee quality data — hover, zoom, and filter.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

from utils.data import load_cqi

df = load_cqi()

print(f"{len(df)} coffees loaded")

1311 coffees loaded


## Altitude vs Cup Score
Hover over points to see farm details.

In [2]:
scatter_df = df.dropna(subset=["altitude_mean_meters", "total_cup_points"])
scatter_df = scatter_df[scatter_df["altitude_mean_meters"] < 5000]

fig = px.scatter(
    scatter_df,
    x="altitude_mean_meters",
    y="total_cup_points",
    color="processing_method",
    hover_data=["farm_name", "country_of_origin", "region", "variety"],
    labels={
        "altitude_mean_meters": "Altitude (m)",
        "total_cup_points": "Total Cup Points",
        "processing_method": "Processing",
    },
    title="Altitude vs Cup Score by Processing Method",
    opacity=0.7,
)
fig.show()

## Average Cup Score by Country (Map)

In [3]:
import pycountry

def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

country_stats = df.groupby("country_of_origin").agg(
    avg_score=("total_cup_points", "mean"),
    sample_count=("total_cup_points", "count"),
).reset_index()

country_stats["iso3"] = country_stats["country_of_origin"].map(to_iso3)
country_stats = country_stats.dropna(subset=["iso3"])

fig = px.choropleth(
    country_stats,
    locations="iso3",
    locationmode="ISO-3",
    color="avg_score",
    hover_name="country_of_origin",
    hover_data=["sample_count"],
    color_continuous_scale="YlOrBr",
    labels={
        "avg_score": "Avg Cup Score",
        "sample_count": "Samples",
        "country_of_origin": "Country",
    },
    title="Average Cup Score by Country",
)
fig.update_layout(geo=dict(showframe=False, projection_type="natural earth"))
fig.show()

## Cupping Score Breakdown by Processing Method
Compare aroma, flavor, acidity, body, etc. across fermentation approaches.

In [4]:
cupping_cols = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance"]
existing = [c for c in cupping_cols if c in df.columns]

method_df = df.dropna(subset=["processing_method"])
melted = method_df.melt(
    id_vars=["processing_method"],
    value_vars=existing,
    var_name="attribute",
    value_name="score",
)

fig = px.box(
    melted,
    x="attribute",
    y="score",
    color="processing_method",
    labels={
        "attribute": "Cupping Attribute",
        "score": "Score",
        "processing_method": "Processing",
    },
    title="Cupping Score Distributions by Processing Method",
)
fig.update_layout(boxmode="group")
fig.show()

## Parallel Coordinates — Cupping Profile by Processing Method
Each line is a coffee. Trace how processing method correlates across all cupping dimensions.

In [5]:
parallel_df = df.dropna(subset=existing + ["processing_method"]).copy()

# Encode processing method as numeric for coloring
methods = parallel_df["processing_method"].unique()
method_map = {m: i for i, m in enumerate(methods)}
parallel_df["method_code"] = parallel_df["processing_method"].map(method_map)

fig = go.Figure(data=go.Parcoords(
    line=dict(
        color=parallel_df["method_code"],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(tickvals=list(method_map.values()), ticktext=list(method_map.keys())),
    ),
    dimensions=[
        dict(label=col.title(), values=parallel_df[col]) for col in existing
    ] + [
        dict(label="Total", values=parallel_df["total_cup_points"]),
    ],
))
fig.update_layout(title="Cupping Profile by Processing Method")
fig.show()

## Top Scoring Coffees
Interactive table of the highest-rated coffees.

In [6]:
top_cols = ["farm_name", "country_of_origin", "region", "altitude",
            "variety", "processing_method", "total_cup_points"]
top_existing = [c for c in top_cols if c in df.columns]

top = df.nlargest(20, "total_cup_points")[top_existing]

fig = go.Figure(data=go.Table(
    header=dict(values=[c.replace("_", " ").title() for c in top_existing]),
    cells=dict(values=[top[c] for c in top_existing]),
))
fig.update_layout(title="Top 20 Coffees by Cup Score")
fig.show()